# Search-R1 电商 Agent · 推理部署压测（Kaggle T4）

**这个 notebook 只干一件事**：量化「Qwen3.5-4B 服务化部署」的性能，产出三个数字。

| 产出 | 对应简历缺口 |
|---|---|
| 并发 1→16 时的吞吐曲线 | 部署/推理加速（403 条 JD 正文提及率 37.2%，Agent 岗 49%） |
| TTFT p50 / p95 尾延迟 | 线上服务 P99 思维 |
| vLLM vs HF `generate` 的加速比 | 会选型、能解释为什么 |

**注意两点**
1. 这里测的是**服务性能**，不是任务准确率。不接检索工具，只压生成。
2. 三个后端请用**同一个 `--max-new-tokens`**（本文件统一 128），否则数字不可比。

**运行方式**：Settings → Accelerator 选 `GPU T4 x2` → Internet **ON** → Save & Run All。

In [ ]:
# Cell 1 · GPU 检查
!nvidia-smi
import torch
assert torch.cuda.is_available(), "请先在右侧 Settings → Accelerator 选择 GPU T4 x2，再重新运行"
props = torch.cuda.get_device_properties(0)
print("GPU        :", torch.cuda.get_device_name(0))
print("显存       :", round(props.total_memory / 1e9, 1), "GB")
print("compute cap:", f"{props.major}.{props.minor}", "(7.5 = T4，只支持 fp16，不支持 bf16)")


In [ ]:
# Cell 2 · 先装「轻依赖」，确保 HF 基线一定跑得起来（不装 vLLM）
import subprocess, sys

def pip_install(*args):
    print("$ pip install", " ".join(args))
    p = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args],
                       capture_output=True, text=True)
    if p.stdout: print(p.stdout[-1200:])
    if p.stderr: print(p.stderr[-1200:])
    print("returncode:", p.returncode)
    return p.returncode == 0

# Qwen3.5 需要 transformers >= 5.x；Kaggle 自带的 4.x 会在加载权重时报架构不识别
ok = pip_install("-U", "transformers>=5.0.0", "accelerate", "openai", "matplotlib")
assert ok, "依赖安装失败，看上面的 stderr"
import transformers
print("transformers:", transformers.__version__)


In [ ]:
# Cell 3 · 拉取仓库
import os, subprocess
os.environ["PYTHONUNBUFFERED"] = "1"
REPO = "/kaggle/working/search-r1-ecommerce-agent"

if not os.path.isdir(REPO):
    r = subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/rayyy032/search-r1-ecommerce-agent.git", REPO],
                       capture_output=True, text=True)
    print("returncode:", r.returncode, r.stderr[-500:])

%cd /kaggle/working/search-r1-ecommerce-agent
!ls
print()
print("压测输入（70 道电商题）:", os.path.exists("datasets/ecommerce/dev.jsonl"))


## 第一步：HF `generate` 基线

先在**没有 vLLM** 的情况下跑，理由有二：① 这条链路只要 `transformers` 就能通，最不容易失败；② vLLM 的加速比必须有一个 baseline 做分母。

`concurrency` 在 HF 模式下等于 **batch size**，不是并发请求数 —— HF 的 `generate` 没有连续批处理，只能整批前向。

In [ ]:
# Cell 4 · HF 基线（约 3-6 分钟）
!python bench_serving.py --backend hf --model Qwen/Qwen3.5-4B \
    --concurrency 1 2 4 8 --requests-per-level 8 --max-new-tokens 128 \
    --output bench_result/hf_baseline.json


## 第二步：装 vLLM，跑离线批处理

`--backend offline` 用 `vllm.LLM` 做离线批量推理，测吞吐上界。

**本轮实测结论：vLLM 在 Kaggle 镜像上装不起来 —— 是环境层面的硬冲突，不是脚本问题。**

- Kaggle 镜像的 CUDA runtime 是 **12.8**，而 `pip install vllm` 拉到的轮子是在 **CUDA 13** 上编译的，
  `import vllm` 直接报 `ImportError: libcudart.so.13: cannot open shared object file`。
- 更麻烦的是它会把 `torch` 一并换成 cu130。这一步之后就**连原本跑通的 HF 基线也会崩**：
  新进程里 torch 是 13.0、torchaudio 还是 12.8，又回到 `compiled with different CUDA versions`。
- Cell 5 的 torchaudio 对齐只解决「torch 与 torchaudio 互不一致」，解决不了
  「vLLM 要 CUDA 13、镜像只有 CUDA 12.8」这个更底层的矛盾。

**所以正确用法是：先把 HF 基线跑完并落盘（Cell 4），再决定要不要试 vLLM。**
HF 数字已经存进 `bench_result/hf_baseline.json`，装 vLLM 失败不影响它。

> 真要跑 vLLM，得换成 CUDA 12.8 编译的旧版轮子并逐个试版本，每次约 10 分钟额度，
> 换来的只是一个加分数字。这笔账不划算，本轮就止步于「HF 基线 + 说明环境原因」。


In [ ]:
# Cell 5 · 安装 vLLM，并**对齐 torch / torchaudio 的 CUDA 版本**
#
# 为什么需要这一段：Kaggle 镜像里 torch 可能是 cu130 构建、torchaudio 却是 cu128。
# 两者 CUDA 版本不一致时，torchaudio/__init__.py 直接抛
#   RuntimeError: Detected that PyTorch and TorchAudio were compiled with different CUDA versions
# 而 transformers 的 audio_utils 是无条件 `import torchaudio` 的，于是 vLLM 连 import 都过不去。
import subprocess, sys, torch

def pip(*args):
    p = subprocess.run([sys.executable, "-m", "pip", *args], capture_output=True, text=True)
    if p.stderr: print(p.stderr[-1000:])
    return p.returncode

print("pip install vllm ->", pip("install", "-q", "vllm"))
print("torch:", torch.__version__, "| torch.version.cuda =", torch.version.cuda)

# ---- 检查 torchaudio 是否因 CUDA 版本不一致而报错 ----
need_fix = False
try:
    import torchaudio
    print("OK  torchaudio 可导入:", torchaudio.__version__)
except Exception as e:
    print("!! torchaudio 导入失败:", type(e).__name__, str(e)[:220])
    need_fix = True

if need_fix and torch.version.cuda:
    tag = "cu" + torch.version.cuda.replace(".", "")   # 例: '13.0' -> 'cu130'
    print(f"   按 torch 的 {tag} 重装 torchaudio ...")
    pip("install", "-q", "--force-reinstall", "torchaudio",
        "--index-url", f"https://download.pytorch.org/whl/{tag}")

# ---- 验收 ----
for label, mod in (("torchaudio", "torchaudio"), ("vllm", "vllm")):
    try:
        m = __import__(mod)
        print(f"OK  {label} 可用:", getattr(m, "__version__", "?"))
    except Exception as e:
        print(f"!! {label} 仍不可用:", type(e).__name__, str(e)[:200])
        if mod == "vllm":
            print("   若实在装不上：用 HF 基线数字 + 说明『vLLM 环境未打通』即可，别伪造数字。")


In [ ]:
# Cell 6 · vLLM 离线批处理
!python bench_serving.py --backend offline --model Qwen/Qwen3.5-4B \
    --concurrency 1 2 4 8 16 --requests-per-level 32 --max-new-tokens 128 \
    --output bench_result/vllm_offline.json


## 第三步：vLLM 起 OpenAI 兼容服务，测 TTFT 与尾延迟

`--backend server` 会自己 `subprocess.Popen(['vllm','serve',...])`、轮询 `/models` 等就绪、逐级压测、结束再关服务。**这一步才是 JD 里说的「部署」**：有首 token 延迟、有 p95 尾延迟。

T4 不支持 bf16，vLLM 会自动用 fp16；4B fp16 约 8GB 显存，T4（16GB）单卡够用。

In [ ]:
# Cell 7 · vLLM server 模式（含 TTFT）
!python bench_serving.py --backend server --model Qwen/Qwen3.5-4B \
    --concurrency 1 2 4 8 16 --requests-per-level 32 --max-new-tokens 128 \
    --port 8000 --output bench_result/vllm_server.json


In [ ]:
# Cell 8 · 汇总所有后端 + 出图
import json
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

files = sorted(Path("bench_result").glob("*.json"))
if not files:
    print("bench_result/ 里没有结果，前面的 cell 没跑成功")
for f in files:
    d = json.loads(f.read_text(encoding="utf-8"))
    print(f"\n=== {f.name} | backend={d['backend']} | model={d['model']} | max_new_tokens={d['max_new_tokens']} ===")
    print(f"{'conc':>5}{'tok/s':>11}{'req/s':>9}{'TTFT_p50':>11}{'TTFT_p95':>11}{'lat_p95':>10}")
    for L in d["levels"]:
        t50, t95 = L.get("ttft_p50_seconds"), L.get("ttft_p95_seconds")
        print(f"{L['concurrency']:>5}{L['throughput_tokens_per_second']:>11.1f}"
              f"{L['throughput_requests_per_second']:>9.2f}"
              f"{(f'{t50:.3f}s' if t50 else '-'):>11}{(f'{t95:.3f}s' if t95 else '-'):>11}"
              f"{L['latency_p95_seconds']:>10.3f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for f in files:
    d = json.loads(f.read_text(encoding="utf-8"))
    x = [L["concurrency"] for L in d["levels"]]
    ax[0].plot(x, [L["throughput_tokens_per_second"] for L in d["levels"]], "o-", label=d["backend"])
    ax[1].plot(x, [L["latency_p95_seconds"] for L in d["levels"]], "o-", label=d["backend"])
ax[0].set_xlabel("concurrency / batch size"); ax[0].set_ylabel("tokens per second")
ax[0].set_title("Throughput scaling"); ax[0].grid(alpha=.3); ax[0].legend()
ax[1].set_xlabel("concurrency / batch size"); ax[1].set_ylabel("p95 latency (s)")
ax[1].set_title("Tail latency"); ax[1].grid(alpha=.3); ax[1].legend()
plt.tight_layout()
plt.savefig("bench_result/throughput_vs_concurrency.png", dpi=140)
print("\nSaved: bench_result/throughput_vs_concurrency.png")
print("下载方式：右侧 Output → 下载整个 working 目录，或 !zip -r /kaggle/working/bench_result.zip bench_result")


## 跑完之后：能交付什么

**本轮（2026-10-10）实际拿到的**：

1. **HF `generate` 基线的吞吐-并发曲线** —— 并发 1/2/4/8 的 `tok/s`
2. **尾延迟** —— p95（注意：是「整批墙钟 ÷ 请求数」的摊销值，不是单请求延迟）
3. **扩展性上界** —— 1→8 路的加速比，说明朴素 HF 部署没有连续批处理，这正是引入 vLLM 的动机

**vLLM 那一档：没跑通，原因已写清楚（见上一节）。**

⚠️ **别做的事**

- 别把 HF 模式下的 `latency` 当成单请求延迟 —— HF 没有连续批处理，那是整批摊销值
- 别在三个后端之间改 `--max-new-tokens`，数字会不可比
- **别在装完 vLLM 之后回到 HF 档重跑** —— torch 已经被换成 cu130，数字不再与新会话可比
- 别伪造数字。vLLM 没装上，就写「HF 基线 + 环境原因」，面试官更认这个

**一个必须知道的坑（本轮已修）**：HF 基线最初用 `AutoTokenizer` 默认的**右填充**做批量生成，
而 decoder-only 模型必须**左填充**，否则模型会从 pad token 之后续写，输出与生成长度都被污染。
同时 T4 不支持 bf16，`dtype="auto"` 会加载 bf16 并退化成慢速实现。
`bench_serving.py` 已改为「左填充 + 显式 fp16」，**因此早期的两轮 HF 数字作废，需用当前版本重跑**。
